# Run the rApp — twin-in-the-loop optimisation on a Virtual RIC

This notebook is the run engine behind the **Run the rApp** artefact page. It executes the
DigitalTwinSim rApp prototypes exactly as described in *Twin-in-the-Loop Optimisation as an
rApp on a Virtual RIC* (full description, v4): the existing configuration is first evaluated
in the twin from the ingested counter dump, candidate changes are rehearsed in the twin, and
only changes that pass the twin gate (§9) are emitted as a configuration intent.

**Engines:** Sionna RT (ray-traced twin, one isotropic trace per tower; every configuration
scored in post with the TR 38.901 pattern) and Sionna SYS (PHYAbstraction link curve and the
slice environment). **Modes:** single tower or cluster (§12). **Functions:** capacity
(twin optimiser), slice / PRB (SYS-scored guarded acceptance), audit (MCS histogram fit,
§10), energy (anchored to the validated RANOpt relationship).

**Version:** artefact v2 - clean run (16 Nov 2026), synced with the v4 full description.

**How to run.** Pick from the dropdowns in the **settings cell** below - MODE ("single" or "cluster"), SITE ("A", "B" or "C", single mode only), FUNCTION ("capacity", "slice" (or "prb"), "energy", "audit" - one step at a time - or "all") and COUNTER_SET ("study", "test1", "test2", "test3" or "operator") - then **Runtime > Run all**. The result - the recommended changes, the before/after numbers and the maps - prints in the **last cell**. The study network (the San Francisco towers and their recorded counters) is built in: nothing to download, upload or paste. Advanced: a package copied from the [artefact page](https://ysrao.github.io/twin-in-the-loop-rapp/rapp-run.html) can be pasted over the package cell to run your own sites and counters instead. The notebook also runs locally (Jupyter on a Mac or an operator's machine): same settings, same Run all, no Google account needed.


In [ ]:
# --- Setup: platform, packages, the rApp modules and the scene ----------
import os, sys, subprocess, urllib.request

import contextlib

@contextlib.contextmanager
def _quiet_backend_probe():
    # Dr.Jit (under Sionna RT) probes its CPU/LLVM backend on import and prints
    # "jitc_llvm_init(): LLVM API initialization failed" when no LLVM library is
    # installed, e.g. on a Mac. It is harmless - the GPU backend (CUDA or Metal)
    # is used - so that native message is muted here. Python errors still show.
    # Dr.Jit logs through Python's sys.stderr, native code through fd 2: mute both.
    import io
    sys.stderr.flush()
    py_err, sys.stderr = sys.stderr, io.StringIO()
    saved, null = os.dup(2), os.open(os.devnull, os.O_WRONLY)
    os.dup2(null, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2); os.close(null); os.close(saved)
        sys.stderr = py_err

def _need(pkg, imp):
    try:
        with _quiet_backend_probe():
            __import__(imp)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

# Test for the submodules actually used: sionna-rt alone provides only
# sionna.rt (a namespace package makes "import sionna" succeed while
# sionna.sys is still missing), so each engine is checked by name.
for pkg, imp in [("sionna-rt==2.2.0", "sionna.rt"), ("sionna==2.2.0", "sionna.sys"),
                 ("trimesh", "trimesh"), ("rtree", "rtree")]:
    _need(pkg, imp)

with _quiet_backend_probe():
    import mitsuba as mi
    import sionna.rt  # picks the ray-tracing backend: NVIDIA GPU (CUDA), Apple GPU (Metal) or CPU (LLVM)
mi.set_log_level(mi.LogLevel.Error)  # hide Mitsuba's "ASCII PLY is slow to parse" warning on the scene file
_backend = {"cuda": "NVIDIA GPU (CUDA)", "metal": "Apple GPU (Metal)", "llvm": "CPU (LLVM)"}
print("ray tracing on:", _backend.get(mi.variant().split("_")[0], mi.variant()), "-", mi.variant())

# Running from a clone of the repository (Jupyter on a Mac or an operator's
# machine): use the clone's own modules and scene. Otherwise (Colab):
# fetch them from GitHub.
_clone = next((d for d in (os.getcwd(), os.path.dirname(os.getcwd()))
               if os.path.exists(os.path.join(d, "rapp", "twin.py"))
               and os.path.exists(os.path.join(d, "data", "city_open.ply"))), None)
if _clone:
    sys.path.insert(0, os.path.join(_clone, "rapp"))
    SCENE = os.path.join(_clone, "data", "city_open.ply")
    print("modules + scene: local clone of the repository")
else:
    REPO_RAW = "https://raw.githubusercontent.com/ysrao/twin-in-the-loop-rapp/main/"
    os.makedirs("rapp", exist_ok=True)
    for f in ["twin.py", "search.py", "cluster.py", "mcs_audit.py", "env_sys.py", "policy.py", "a1_check.py", "o1_check.py"]:
        urllib.request.urlretrieve(REPO_RAW + "rapp/" + f, f"rapp/{f}")  # always fresh: a persistent runtime must not keep stale modules
    os.makedirs("rapp/yang", exist_ok=True)
    urllib.request.urlretrieve(REPO_RAW + "rapp/yang/twin-rapp-vendor-cm@2026-11-17.yang",
                                 "rapp/yang/twin-rapp-vendor-cm@2026-11-17.yang")  # the O1 check's vendor module
    sys.path.insert(0, os.path.abspath("rapp"))
    SCENE = "city_open.ply"
    if not os.path.exists(SCENE):
        urllib.request.urlretrieve(REPO_RAW + "data/city_open.ply", SCENE)
    print("modules + scene: downloaded from GitHub")
print("scene ready:", os.path.getsize(SCENE), "bytes")

In [ ]:
#@title What to run - pick from the dropdowns, then Run all
# --- What to run ---------------------------------------------------------
# Change these settings, then Run all. That is all a reader has to do:
# the study network (the San Francisco towers and their recorded
# counters) is already inside this notebook.
MODE = "single" #@param ["single", "cluster"]
# "single" = one tower (SITE) | "cluster" = Towers A + B + C
SITE = "B" #@param ["A", "B", "C"]
FUNCTION = "capacity" #@param ["capacity", "slice", "energy", "audit", "all"]
# run ONE step: "capacity" | "slice" (= "prb") | "energy" | "audit" - or "all"
# Advanced: to run your own sites and counters instead, build a package
# on the artefact page, click "Copy the package" there, and paste it over
# the whole of the package cell below - a pasted package overrides these
# settings (its own mode, function and counters are used).
COUNTER_SET = "study" #@param ["study", "test1", "test2", "test3", "operator"]
A1_CHECK = True #@param {type:"boolean"}
O1_CHECK = True #@param {type:"boolean"}
# A1_CHECK: send the slice and energy decisions as A1 policies to the O-RAN SC A1
# simulator (set up and started by the notebook, on this machine only).
# O1_CHECK: write the configuration changes (tilt, azimuth, power, OLLA, MCS cap, P0)
# as O1 edits and check each against the 3GPP NR YANG models (or a vendor-style module).
# The counter set to judge the existing configuration by:
#   "study"    - the study network's recorded counters (the default)
#   "test1"    - Test data 1: a loaded cell, cell-edge suffering
#   "test2"    - Test data 2: a quiet cell, many low-traffic hours
#   "test3"    - Test data 3: an edge-poor cell, low fairness
#   "operator" - Operator added: your own counter dump, edited in below
OPERATOR_COUNTERS = {
    "median_dl": 38.0, "median_ul": 11.0, "edge_dl": 6.5, "edge_ul": 2.0,
    "fairness": 0.41, "prb_util": 81.0, "signalling": 34.0, "load": 0.75,
    "low_hours": 6, "carriers_per_site": 2, "current_shares": [0.6, 0.2, 0.2],
}  # <- operator: replace these numbers with your own counter dump
PM_CSV = ""  # MCS audit input: "" = the built-in MCS histograms, or a CSV file path,
# e.g. PM_CSV = "../data/test_pm_mcs.csv" (paths are relative to this notebook's folder).
# One row per sector and MCS index, with these columns:
#   site_id, sector (1-3), mcs (0-27), grant_share, olla_offset_db, mcs_cap
# grant_share is each MCS index's share of the sector's grants (normalised per sector);
# olla_offset_db and mcs_cap are the gNB's configured values (same on every row).


In [ ]:
# --- The run package: sites + existing config + counters + function ----
# This cell is ONLY the package. On the artefact page, choose the mode and
# the function and click "Copy the package" - the copied text is this whole
# cell. Click this cell, select all (Cmd/Ctrl+A) and paste over it.
# Leave the cell as it is to run the built-in study case.
PACKAGE_JSON = r'''
{
 "mode": "single",
 "function": "capacity",
 "sites": [
  {
   "site_id": "sf-twin-tower-B",
   "label": "Tower B",
   "position_enu_m": [
    250.0,
    250.0
   ],
   "ground_z_m": -64.75,
   "height_m_agl": 30.0,
   "azimuths_deg": [
    90.0,
    210.0,
    330.0
   ],
   "tilts_deg": [
    10.0,
    0.97,
    3.92
   ],
   "power_offset_db": 0.0,
   "traffic_shares_current": [
    0.4,
    0.35,
    0.25
   ],
   "eval_half_m": 150.0,
   "eval_step_m": 20.0
  }
 ],
 "site_traffic_totals": {
  "sf-twin-tower-B": 1.0
 },
 "policy": {
  "traffic_shares_projected": [
   0.34,
   0.3,
   0.36
  ],
  "min_coverage_delta_pts": 0.0,
  "max_edge_drop_db": 1.0
 },
 "counters": {
  "median_dl": 38.0,
  "median_ul": 11.0,
  "edge_dl": 6.5,
  "edge_ul": 2.0,
  "fairness": 0.41,
  "prb_util": 81.0,
  "signalling": 34.0,
  "load": 0.75,
  "low_hours": 6,
  "carriers_per_site": 2,
  "current_shares": [
   0.6,
   0.2,
   0.2
  ]
 },
 "pm": {
  "site_id": "sf-twin-tower-B",
  "gnb_configured": {
   "olla_offset_db": 0.0,
   "mcs_cap": 24
  },
  "mcs_histograms_by_sector": {
   "0": [
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.04395,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0219,
    0.0,
    0.06445,
    0.0,
    0.0,
    0.0203,
    0.17165,
    0.0444,
    0.2833,
    0.0,
    0.3042,
    0.04585,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0
   ],
   "1": [
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.02455,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.09245,
    0.0,
    0.0,
    0.0466,
    0.04535,
    0.069,
    0.21025,
    0.0,
    0.2549,
    0.2569,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0
   ],
   "2": [
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0381,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0777,
    0.0,
    0.0,
    0.11125,
    0.07655,
    0.07875,
    0.11735,
    0.0,
    0.2711,
    0.194,
    0.0352,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0
   ]
  }
 }
}
'''


In [ ]:
import json as _json, copy as _copy
try:
    _pasted = _json.loads(PACKAGE_JSON)
except Exception:
    raise ValueError("The package cell does not hold valid JSON - if you pasted a "
                     "package from the artefact page, the paste went wrong: click "
                     "'Copy the package' there, click the package cell here, select "
                     "all (Cmd/Ctrl+A) and paste over the whole cell, then Run all again.")
STUDY_PACKAGES = {"single": {"A": {'mode': 'single', 'function': 'capacity', 'sites': [{'site_id': 'sf-twin-tower-A', 'label': 'Tower A', 'position_enu_m': [-250, 250], 'ground_z_m': -12.67, 'height_m_agl': 30, 'azimuths_deg': [90, 210, 330], 'tilts_deg': [7.9, 6.45, 10], 'power_offset_db': 0, 'traffic_shares_current': [0.38, 0.34, 0.28], 'eval_half_m': 150, 'eval_step_m': 20}], 'site_traffic_totals': {'sf-twin-tower-A': 1.0}, 'policy': {'traffic_shares_projected': [0.34, 0.3, 0.36], 'min_coverage_delta_pts': 0, 'max_edge_drop_db': 1}, 'counters': {'median_dl': 38, 'median_ul': 11, 'edge_dl': 6.5, 'edge_ul': 2, 'fairness': 0.41, 'prb_util': 81, 'signalling': 34, 'load': 0.75, 'low_hours': 6, 'carriers_per_site': 2, 'current_shares': [0.6, 0.2, 0.2]}, 'pm': {'site_id': 'sf-twin-tower-B', 'gnb_configured': {'olla_offset_db': 0, 'mcs_cap': 24}, 'mcs_histograms_by_sector': {'0': [0, 0, 0, 0, 0, 0.04395, 0, 0, 0, 0, 0.0219, 0, 0.06445, 0, 0, 0.0203, 0.17165, 0.0444, 0.2833, 0, 0.3042, 0.04585, 0, 0, 0, 0, 0, 0], '1': [0, 0, 0, 0, 0, 0.02455, 0, 0, 0, 0, 0, 0, 0.09245, 0, 0, 0.0466, 0.04535, 0.069, 0.21025, 0, 0.2549, 0.2569, 0, 0, 0, 0, 0, 0], '2': [0, 0, 0, 0, 0, 0.0381, 0, 0, 0, 0, 0, 0, 0.0777, 0, 0, 0.11125, 0.07655, 0.07875, 0.11735, 0, 0.2711, 0.194, 0.0352, 0, 0, 0, 0, 0]}}}, "B": {'mode': 'single', 'function': 'capacity', 'sites': [{'site_id': 'sf-twin-tower-B', 'label': 'Tower B', 'position_enu_m': [250, 250], 'ground_z_m': -64.75, 'height_m_agl': 30, 'azimuths_deg': [90, 210, 330], 'tilts_deg': [10, 0.97, 3.92], 'power_offset_db': 0, 'traffic_shares_current': [0.4, 0.35, 0.25], 'eval_half_m': 150, 'eval_step_m': 20}], 'site_traffic_totals': {'sf-twin-tower-B': 1}, 'policy': {'traffic_shares_projected': [0.34, 0.3, 0.36], 'min_coverage_delta_pts': 0, 'max_edge_drop_db': 1}, 'counters': {'median_dl': 38, 'median_ul': 11, 'edge_dl': 6.5, 'edge_ul': 2, 'fairness': 0.41, 'prb_util': 81, 'signalling': 34, 'load': 0.75, 'low_hours': 6, 'carriers_per_site': 2, 'current_shares': [0.6, 0.2, 0.2]}, 'pm': {'site_id': 'sf-twin-tower-B', 'gnb_configured': {'olla_offset_db': 0, 'mcs_cap': 24}, 'mcs_histograms_by_sector': {'0': [0, 0, 0, 0, 0, 0.04395, 0, 0, 0, 0, 0.0219, 0, 0.06445, 0, 0, 0.0203, 0.17165, 0.0444, 0.2833, 0, 0.3042, 0.04585, 0, 0, 0, 0, 0, 0], '1': [0, 0, 0, 0, 0, 0.02455, 0, 0, 0, 0, 0, 0, 0.09245, 0, 0, 0.0466, 0.04535, 0.069, 0.21025, 0, 0.2549, 0.2569, 0, 0, 0, 0, 0, 0], '2': [0, 0, 0, 0, 0, 0.0381, 0, 0, 0, 0, 0, 0, 0.0777, 0, 0, 0.11125, 0.07655, 0.07875, 0.11735, 0, 0.2711, 0.194, 0.0352, 0, 0, 0, 0, 0]}}}, "C": {'mode': 'single', 'function': 'capacity', 'sites': [{'site_id': 'sf-twin-tower-C', 'label': 'Tower C', 'position_enu_m': [0, -250], 'ground_z_m': -38.89, 'height_m_agl': 30, 'azimuths_deg': [90, 210, 330], 'tilts_deg': [7.59, 7.68, 3.23], 'power_offset_db': 0, 'traffic_shares_current': [0.36, 0.33, 0.31], 'eval_half_m': 150, 'eval_step_m': 20}], 'site_traffic_totals': {'sf-twin-tower-C': 1.0}, 'policy': {'traffic_shares_projected': [0.34, 0.3, 0.36], 'min_coverage_delta_pts': 0, 'max_edge_drop_db': 1}, 'counters': {'median_dl': 38, 'median_ul': 11, 'edge_dl': 6.5, 'edge_ul': 2, 'fairness': 0.41, 'prb_util': 81, 'signalling': 34, 'load': 0.75, 'low_hours': 6, 'carriers_per_site': 2, 'current_shares': [0.6, 0.2, 0.2]}, 'pm': {'site_id': 'sf-twin-tower-B', 'gnb_configured': {'olla_offset_db': 0, 'mcs_cap': 24}, 'mcs_histograms_by_sector': {'0': [0, 0, 0, 0, 0, 0.04395, 0, 0, 0, 0, 0.0219, 0, 0.06445, 0, 0, 0.0203, 0.17165, 0.0444, 0.2833, 0, 0.3042, 0.04585, 0, 0, 0, 0, 0, 0], '1': [0, 0, 0, 0, 0, 0.02455, 0, 0, 0, 0, 0, 0, 0.09245, 0, 0, 0.0466, 0.04535, 0.069, 0.21025, 0, 0.2549, 0.2569, 0, 0, 0, 0, 0, 0], '2': [0, 0, 0, 0, 0, 0.0381, 0, 0, 0, 0, 0, 0, 0.0777, 0, 0, 0.11125, 0.07655, 0.07875, 0.11735, 0, 0.2711, 0.194, 0.0352, 0, 0, 0, 0, 0]}}}}, "cluster": {'mode': 'cluster', 'function': 'capacity', 'sites': [{'site_id': 'sf-twin-tower-A', 'label': 'Tower A', 'position_enu_m': [-250, 250], 'ground_z_m': -12.67, 'height_m_agl': 30, 'azimuths_deg': [90, 210, 330], 'tilts_deg': [7.9, 6.45, 10], 'power_offset_db': 0, 'traffic_shares_current': [0.38, 0.34, 0.28], 'eval_half_m': 150, 'eval_step_m': 20}, {'site_id': 'sf-twin-tower-B', 'label': 'Tower B', 'position_enu_m': [250, 250], 'ground_z_m': -64.75, 'height_m_agl': 30, 'azimuths_deg': [90, 210, 330], 'tilts_deg': [10, 0.97, 3.92], 'power_offset_db': 0, 'traffic_shares_current': [0.4, 0.35, 0.25], 'eval_half_m': 150, 'eval_step_m': 20}, {'site_id': 'sf-twin-tower-C', 'label': 'Tower C', 'position_enu_m': [0, -250], 'ground_z_m': -38.89, 'height_m_agl': 30, 'azimuths_deg': [90, 210, 330], 'tilts_deg': [7.59, 7.68, 3.23], 'power_offset_db': 0, 'traffic_shares_current': [0.36, 0.33, 0.31], 'eval_half_m': 150, 'eval_step_m': 20}], 'site_traffic_totals': {'sf-twin-tower-A': 0.3, 'sf-twin-tower-B': 0.4, 'sf-twin-tower-C': 0.3}, 'policy': {'traffic_shares_projected': {'sf-twin-tower-A': [0.34, 0.33, 0.33], 'sf-twin-tower-B': [0.34, 0.3, 0.36], 'sf-twin-tower-C': [0.32, 0.32, 0.36]}, 'min_coverage_delta_pts': 0, 'max_edge_drop_db': 1}, 'counters': {'median_dl': 38, 'median_ul': 11, 'edge_dl': 6.5, 'edge_ul': 2, 'fairness': 0.41, 'prb_util': 81, 'signalling': 34, 'load': 0.75, 'low_hours': 6, 'carriers_per_site': 2, 'current_shares': [0.6, 0.2, 0.2]}, 'pm': {'site_id': 'sf-twin-tower-B', 'gnb_configured': {'olla_offset_db': 0, 'mcs_cap': 24}, 'mcs_histograms_by_sector': {'0': [0, 0, 0, 0, 0, 0.04395, 0, 0, 0, 0, 0.0219, 0, 0.06445, 0, 0, 0.0203, 0.17165, 0.0444, 0.2833, 0, 0.3042, 0.04585, 0, 0, 0, 0, 0, 0], '1': [0, 0, 0, 0, 0, 0.02455, 0, 0, 0, 0, 0, 0, 0.09245, 0, 0, 0.0466, 0.04535, 0.069, 0.21025, 0, 0.2549, 0.2569, 0, 0, 0, 0, 0, 0], '2': [0, 0, 0, 0, 0, 0.0381, 0, 0, 0, 0, 0, 0, 0.0777, 0, 0, 0.11125, 0.07655, 0.07875, 0.11735, 0, 0.2711, 0.194, 0.0352, 0, 0, 0, 0, 0]}}}}
if _pasted == STUDY_PACKAGES["single"]["B"]:
    _mode = str(MODE).strip().lower()
    _fn = str(FUNCTION).strip().lower()
    _fn = {"prb": "slice", "slice/prb": "slice", "slices": "slice"}.get(_fn, _fn)  # "prb" = the slice / PRB step
    if _mode not in STUDY_PACKAGES:
        raise ValueError('MODE must be "single" or "cluster" - the settings cell has %r.' % (MODE,))
    _site = str(SITE).strip().upper()
    if _mode == "single" and _site not in STUDY_PACKAGES["single"]:
        raise ValueError('SITE must be "A", "B" or "C" - the settings cell has %r.' % (SITE,))
    if _fn not in ("capacity", "slice", "energy", "audit", "all"):
        raise ValueError('FUNCTION must be "capacity", "slice" (or "prb"), "energy", "audit" or "all" - the settings cell has %r.' % (FUNCTION,))
    PKG = _copy.deepcopy(STUDY_PACKAGES[_mode][_site] if _mode == "single" else STUDY_PACKAGES[_mode])
    PKG["function"] = _fn
    print("settings cell: study network,", _mode, "mode," + ((" Tower " + _site + ",") if _mode == "single" else ""), _fn, "function")
    TEST_COUNTERS = {
        "test1": {"median_dl": 52.0, "median_ul": 17.0, "edge_dl": 4.1, "edge_ul": 1.2,
                  "fairness": 0.33, "prb_util": 93.0, "signalling": 41.0, "load": 0.92,
                  "low_hours": 3, "carriers_per_site": 3, "current_shares": [0.7, 0.15, 0.15]},
        "test2": {"median_dl": 18.0, "median_ul": 6.0, "edge_dl": 9.0, "edge_ul": 3.0,
                  "fairness": 0.71, "prb_util": 44.0, "signalling": 22.0, "load": 0.35,
                  "low_hours": 9, "carriers_per_site": 2, "current_shares": [0.5, 0.25, 0.25]},
        "test3": {"median_dl": 29.0, "median_ul": 9.0, "edge_dl": 2.8, "edge_ul": 0.9,
                  "fairness": 0.28, "prb_util": 77.0, "signalling": 38.0, "load": 0.68,
                  "low_hours": 5, "carriers_per_site": 2, "current_shares": [0.55, 0.3, 0.15]},
    }
    _cs = str(COUNTER_SET).strip().lower()
    if _cs == "operator":
        PKG["counters"] = _copy.deepcopy(OPERATOR_COUNTERS)
        print("counters: Operator added (the OPERATOR_COUNTERS in the settings cell)")
    elif _cs in TEST_COUNTERS:
        PKG["counters"] = _copy.deepcopy(TEST_COUNTERS[_cs])
        print("counters: Test data", _cs[-1], "(built-in test set)")
    elif _cs != "study":
        raise ValueError('COUNTER_SET must be "study", "test1", "test2", "test3" or "operator" - the settings cell has %r.' % (COUNTER_SET,))
    else:
        # Per-tower counters: Tower B the study network's recorded counters, Towers A
        # and C dummy values in line with their link counters (both near 55 % PRB use).
        STUDY_COUNTERS_BY_SITE = {
            "sf-twin-tower-A": {"median_dl": 25.0, "median_ul": 9.0, "edge_dl": 4.2, "edge_ul": 1.6,
                                "fairness": 0.38, "prb_util": 55.0, "signalling": 26.0, "load": 0.55,
                                "low_hours": 8, "carriers_per_site": 2, "current_shares": [0.6, 0.2, 0.2]},
            "sf-twin-tower-B": _copy.deepcopy(PKG["counters"]),
            "sf-twin-tower-C": {"median_dl": 26.0, "median_ul": 6.0, "edge_dl": 5.5, "edge_ul": 0.9,
                                "fairness": 0.44, "prb_util": 55.0, "signalling": 24.0, "load": 0.55,
                                "low_hours": 8, "carriers_per_site": 2, "current_shares": [0.6, 0.2, 0.2]},
        }
        PKG["counters_by_site"] = {s["site_id"]: STUDY_COUNTERS_BY_SITE[s["site_id"]]
                                   for s in PKG["sites"] if s["site_id"] in STUDY_COUNTERS_BY_SITE}
        print("counters: per tower - Tower B the study network's recorded counters, "
              "Towers A and C dummy values")
    # Dummy MCS histograms for Towers A and C (data/test_pm_mcs_tower_A.csv / _C.csv),
    # so the audit covers every tower of the run, not only Tower B.
    PKG["pm_sites"] = [PKG["pm"]] + [{"site_id": "sf-twin-tower-A", "gnb_configured": {"olla_offset_db": 0.0, "mcs_cap": 24}, "mcs_histograms_by_sector": {"0": [0.0, 4e-05, 0.00016, 0.00051, 0.00148, 0.0038, 0.00875, 0.01801, 0.03318, 0.05471, 0.08072, 0.10656, 0.12589, 0.13308, 0.12589, 0.10656, 0.08072, 0.05471, 0.03318, 0.01801, 0.00875, 0.0038, 0.00148, 0.0, 0.0, 0.0, 0.0, 0.0], "1": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1e-05, 6e-05, 0.00027, 0.00104, 0.00344, 0.00973, 0.02346, 0.04819, 0.08437, 0.12586, 0.16, 0.17333, 0.16, 0.12586, 0.08437, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "2": [0.0, 0.0, 0.01567, 0.02664, 0.04174, 0.06027, 0.08021, 0.09837, 0.11118, 0.11581, 0.11118, 0.09837, 0.08021, 0.06027, 0.04174, 0.02664, 0.01567, 0.0085, 0.00425, 0.00195, 0.00083, 0.00032, 0.00012, 4e-05, 0.0, 0.0, 0.0, 0.0]}}, {"site_id": "sf-twin-tower-C", "gnb_configured": {"olla_offset_db": -1.0, "mcs_cap": 24}, "mcs_histograms_by_sector": {"0": [0.0, 0.0, 0.0, 0.0, 4e-05, 0.00016, 0.00051, 0.00148, 0.0038, 0.00875, 0.01801, 0.03318, 0.05471, 0.08072, 0.10656, 0.12589, 0.13308, 0.12589, 0.10656, 0.08072, 0.05471, 0.03318, 0.01801, 0.00875, 0.0038, 0.00148, 0.0, 0.0], "1": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1e-05, 6e-05, 0.00027, 0.00104, 0.00344, 0.00973, 0.02346, 0.04819, 0.08437, 0.12586, 0.16, 0.17333, 0.16, 0.12586, 0.08437, 0.0, 0.0, 0.0], "2": [0.0, 0.0, 0.0, 0.0, 0.0, 0.01567, 0.02664, 0.04174, 0.06027, 0.08021, 0.09837, 0.11118, 0.11581, 0.11118, 0.09837, 0.08021, 0.06027, 0.04174, 0.02664, 0.01567, 0.0085, 0.00425, 0.00195, 0.00083, 0.00032, 0.00012, 4e-05, 0.0]}}]
    # Dummy daily link counters (BLER, HARQ, rank, power control, load) for A, B, C,
    # aggregated per cell from data/pm_synth/pm_daily_counters.csv + site_config.csv.
    PKG["link_pm"] = [{"site_id": "sf-twin-tower-A", "sectors": {"1": {"config": {"dl_bler_target_pct": 10.0, "p0_nominal_pusch_dbm": -104.0, "pusch_alpha": 0.8, "mimo_layers_max": 4}, "days": 62, "median": {"prb_util_dl_pct": 54.615, "user_dl_tput_mbps": 24.625, "avg_cqi": 8.5, "avg_dl_mcs": 14.85, "dl_bler_initial_pct": 16.435, "dl_bler_residual_pct": 1.285, "harq_dl_retx_rate_pct": 19.145, "harq_dl_avg_tx": 1.18, "ul_bler_initial_pct": 9.58, "dl_rank1_pct": 26.0, "dl_avg_rank": 2.24, "ul_sinr_avg_db": 11.4, "ul_phr_avg_db": 15.2, "ul_phr_negative_pct": 9.15, "pusch_tx_power_avg_dbm": 9.2}, "congested_days_pct": 0.0}, "2": {"config": {"dl_bler_target_pct": 10.0, "p0_nominal_pusch_dbm": -104.0, "pusch_alpha": 0.8, "mimo_layers_max": 4}, "days": 62, "median": {"prb_util_dl_pct": 56.525, "user_dl_tput_mbps": 26.415, "avg_cqi": 8.7, "avg_dl_mcs": 15.25, "dl_bler_initial_pct": 21.19, "dl_bler_residual_pct": 2.225, "harq_dl_retx_rate_pct": 25.68, "harq_dl_avg_tx": 1.239, "ul_bler_initial_pct": 9.405, "dl_rank1_pct": 25.65, "dl_avg_rank": 2.23, "ul_sinr_avg_db": 11.0, "ul_phr_avg_db": 14.85, "ul_phr_negative_pct": 8.05, "pusch_tx_power_avg_dbm": 8.8}, "congested_days_pct": 0.0}, "3": {"config": {"dl_bler_target_pct": 10.0, "p0_nominal_pusch_dbm": -104.0, "pusch_alpha": 0.8, "mimo_layers_max": 4}, "days": 62, "median": {"prb_util_dl_pct": 56.535, "user_dl_tput_mbps": 24.405, "avg_cqi": 8.55, "avg_dl_mcs": 15.0, "dl_bler_initial_pct": 16.0, "dl_bler_residual_pct": 1.4, "harq_dl_retx_rate_pct": 18.555, "harq_dl_avg_tx": 1.175, "ul_bler_initial_pct": 9.63, "dl_rank1_pct": 26.2, "dl_avg_rank": 2.23, "ul_sinr_avg_db": 11.15, "ul_phr_avg_db": 14.65, "ul_phr_negative_pct": 8.3, "pusch_tx_power_avg_dbm": 9.2}, "congested_days_pct": 0.0}}}, {"site_id": "sf-twin-tower-B", "sectors": {"1": {"config": {"dl_bler_target_pct": 10.0, "p0_nominal_pusch_dbm": -104.0, "pusch_alpha": 0.8, "mimo_layers_max": 4}, "days": 62, "median": {"prb_util_dl_pct": 83.495, "user_dl_tput_mbps": 10.12, "avg_cqi": 10.4, "avg_dl_mcs": 18.25, "dl_bler_initial_pct": 10.385, "dl_bler_residual_pct": 0.345, "harq_dl_retx_rate_pct": 11.46, "harq_dl_avg_tx": 1.111, "ul_bler_initial_pct": 9.415, "dl_rank1_pct": 30.5, "dl_avg_rank": 2.09, "ul_sinr_avg_db": 11.3, "ul_phr_avg_db": 14.85, "ul_phr_negative_pct": 9.5, "pusch_tx_power_avg_dbm": 9.35}, "congested_days_pct": 46.8}, "2": {"config": {"dl_bler_target_pct": 10.0, "p0_nominal_pusch_dbm": -104.0, "pusch_alpha": 0.8, "mimo_layers_max": 4}, "days": 62, "median": {"prb_util_dl_pct": 88.51, "user_dl_tput_mbps": 9.365, "avg_cqi": 10.4, "avg_dl_mcs": 18.1, "dl_bler_initial_pct": 10.215, "dl_bler_residual_pct": 0.34, "harq_dl_retx_rate_pct": 11.26, "harq_dl_avg_tx": 1.109, "ul_bler_initial_pct": 9.425, "dl_rank1_pct": 29.7, "dl_avg_rank": 2.095, "ul_sinr_avg_db": 10.9, "ul_phr_avg_db": 14.4, "ul_phr_negative_pct": 9.1, "pusch_tx_power_avg_dbm": 9.35}, "congested_days_pct": 74.2}, "3": {"config": {"dl_bler_target_pct": 10.0, "p0_nominal_pusch_dbm": -104.0, "pusch_alpha": 0.8, "mimo_layers_max": 4}, "days": 62, "median": {"prb_util_dl_pct": 84.875, "user_dl_tput_mbps": 9.26, "avg_cqi": 10.4, "avg_dl_mcs": 18.3, "dl_bler_initial_pct": 9.91, "dl_bler_residual_pct": 0.38, "harq_dl_retx_rate_pct": 10.89, "harq_dl_avg_tx": 1.105, "ul_bler_initial_pct": 9.535, "dl_rank1_pct": 30.55, "dl_avg_rank": 2.1, "ul_sinr_avg_db": 10.4, "ul_phr_avg_db": 14.45, "ul_phr_negative_pct": 9.45, "pusch_tx_power_avg_dbm": 9.3}, "congested_days_pct": 50.0}}}, {"site_id": "sf-twin-tower-C", "sectors": {"1": {"config": {"dl_bler_target_pct": 10.0, "p0_nominal_pusch_dbm": -96.0, "pusch_alpha": 0.8, "mimo_layers_max": 4}, "days": 62, "median": {"prb_util_dl_pct": 54.135, "user_dl_tput_mbps": 26.59, "avg_cqi": 11.3, "avg_dl_mcs": 19.9, "dl_bler_initial_pct": 10.04, "dl_bler_residual_pct": 0.34, "harq_dl_retx_rate_pct": 11.045, "harq_dl_avg_tx": 1.107, "ul_bler_initial_pct": 13.815, "dl_rank1_pct": 65.25, "dl_avg_rank": 1.47, "ul_sinr_avg_db": 6.05, "ul_phr_avg_db": 3.4, "ul_phr_negative_pct": 33.9, "pusch_tx_power_avg_dbm": 19.4}, "congested_days_pct": 0.0}, "2": {"config": {"dl_bler_target_pct": 10.0, "p0_nominal_pusch_dbm": -96.0, "pusch_alpha": 0.8, "mimo_layers_max": 4}, "days": 62, "median": {"prb_util_dl_pct": 55.405, "user_dl_tput_mbps": 28.07, "avg_cqi": 11.35, "avg_dl_mcs": 19.65, "dl_bler_initial_pct": 9.8, "dl_bler_residual_pct": 0.35, "harq_dl_retx_rate_pct": 10.755, "harq_dl_avg_tx": 1.104, "ul_bler_initial_pct": 16.62, "dl_rank1_pct": 68.2, "dl_avg_rank": 1.44, "ul_sinr_avg_db": 4.65, "ul_phr_avg_db": 1.6, "ul_phr_negative_pct": 40.8, "pusch_tx_power_avg_dbm": 19.3}, "congested_days_pct": 0.0}, "3": {"config": {"dl_bler_target_pct": 10.0, "p0_nominal_pusch_dbm": -96.0, "pusch_alpha": 0.8, "mimo_layers_max": 4}, "days": 62, "median": {"prb_util_dl_pct": 54.46, "user_dl_tput_mbps": 24.635, "avg_cqi": 11.3, "avg_dl_mcs": 19.95, "dl_bler_initial_pct": 9.905, "dl_bler_residual_pct": 0.365, "harq_dl_retx_rate_pct": 10.89, "harq_dl_avg_tx": 1.105, "ul_bler_initial_pct": 14.17, "dl_rank1_pct": 65.4, "dl_avg_rank": 1.48, "ul_sinr_avg_db": 6.05, "ul_phr_avg_db": 3.8, "ul_phr_negative_pct": 31.8, "pusch_tx_power_avg_dbm": 19.6}, "congested_days_pct": 0.0}}}]
else:
    PKG = _pasted
    print("package: pasted from the artefact page")
# --- Optional: MCS histograms from a PM CSV (the PM_CSV setting) ---------
_pm_csv = str(globals().get("PM_CSV", "") or "").strip()
if _pm_csv:
    import csv as _csv, os as _os
    if not _os.path.exists(_pm_csv):
        raise FileNotFoundError(f"PM_CSV: no file at {_pm_csv!r} (looked in {_os.getcwd()}). "
                                "Use a path relative to this notebook's folder, e.g. '../data/test_pm_mcs.csv'.")
    with open(_pm_csv, newline="") as _f:
        _rows = list(_csv.DictReader(_f))
    _need_cols = {"site_id", "sector", "mcs", "grant_share", "olla_offset_db", "mcs_cap"}
    if not _rows or not _need_cols <= set(_rows[0]):
        raise ValueError(f"PM_CSV {_pm_csv!r} needs the columns {sorted(_need_cols)}; "
                         f"it has {sorted(_rows[0]) if _rows else 'no rows'}.")
    # One file may hold one tower or several (one block of rows per site_id).
    _run_ids = [s["site_id"] for s in PKG["sites"]]
    _sites = sorted({r["site_id"] for r in _rows})
    _bad = [x for x in _sites if x not in _run_ids]
    if _bad:
        raise ValueError(f"PM_CSV site_id(s) {_bad} are not sites of this run ({_run_ids}).")
    _pms = []
    for _sid in _sites:
        _sr = [r for r in _rows if r["site_id"] == _sid]
        _h = {str(s): [0.0] * 28 for s in range(3)}
        for _r in _sr:
            _sec, _m = int(_r["sector"]) - 1, int(_r["mcs"])
            if _sec not in (0, 1, 2) or not 0 <= _m <= 27:
                raise ValueError(f"PM_CSV row has sector {_r['sector']} / mcs {_r['mcs']}: sector must be 1-3, mcs 0-27.")
            _h[str(_sec)][_m] += float(_r["grant_share"])
        for _s, _v in _h.items():
            _tot = sum(_v)
            if _tot <= 0:
                raise ValueError(f"PM_CSV has no grants for {_sid} sector {int(_s) + 1}.")
            _h[_s] = [x / _tot for x in _v]
        _pms.append({"site_id": _sid,
                     "gnb_configured": {"olla_offset_db": float(_sr[0]["olla_offset_db"]),
                                        "mcs_cap": int(float(_sr[0]["mcs_cap"]))},
                     "mcs_histograms_by_sector": _h})
    PKG["pm"] = _pms[0]
    PKG["pm_sites"] = _pms
    print(f"MCS histograms: from {_pm_csv} ({len(_rows)} rows, sites {_sites})")
MODE = PKG.get("mode", "single")
FUNCTION = PKG.get("function", "capacity")
print("mode:", MODE, "| function:", FUNCTION, "| sites:", [s["site_id"] for s in PKG["sites"]])
_cbs = PKG.get("counters_by_site") or {}
print("counters:", "per tower for " + ", ".join(k.replace("sf-twin-tower-", "Tower ") for k in _cbs)
      if _cbs else "one set shared by every tower of the run")

# --- Shared helpers -------------------------------------------------------
import numpy as np
import twin as twin_mod
from search import optimise
from cluster import ClusterScorer, optimise_cluster

PLY = SCENE
RESULT = {"mode": MODE, "function": FUNCTION, "sites": [s["site_id"] for s in PKG["sites"]],
          "changes": [], "notes": []}
_traces = {}

def trace_for(site, rx, tag):
    key = (site["site_id"], tag)
    if key not in _traces:
        tx = [site["position_enu_m"][0], site["position_enu_m"][1],
              site["ground_z_m"] + site["height_m_agl"]]
        _traces[key] = twin_mod.trace_tower(PLY, tx, rx)
        print(f"trace {site['site_id']} ({tag}): {_traces[key]['trace_s']:.1f}s, {len(rx)} receivers")
    return _traces[key]

def counters_for(site_id):
    """This tower's counters: counters_by_site[site_id] if the package has it, else the shared set."""
    return (PKG.get("counters_by_site") or {}).get(site_id) or PKG["counters"]

def site_label(site_id):
    return next((s.get("label", site_id) for s in PKG["sites"] if s["site_id"] == site_id), site_id)

def cfg_of(site):
    return {"azimuths": list(site["azimuths_deg"]), "tilts": list(site["tilts_deg"]),
            "power_offset_db": float(site.get("power_offset_db", 0.0))}

def fmt_k(k):
    return (f"coverage {k['coverage_pct']:.1f}%  median RSRP {k['median_rsrp']:.1f} dBm  "
            f"edge p5 {k['edge_rsrp_p5']:.1f}  median SINR {k['median_sinr']:.1f} dB  "
            f"weighted SE {k['capacity_weighted_se']:.2f} bit/s/Hz")

def describe_changes(before, after):
    out = []
    for sid in before:
        b, a = before[sid], after[sid]
        lbl = sid.replace("sf-twin-tower-", "Tower ")
        for s in range(3):
            if a["azimuths"][s] != b["azimuths"][s]:
                out.append(f"{lbl} sector {s+1}: azimuth {b['azimuths'][s]:.0f} -> {a['azimuths'][s]:.0f} deg")
            if a["tilts"][s] != b["tilts"][s]:
                out.append(f"{lbl} sector {s+1}: tilt {b['tilts'][s]:.2f} -> {a['tilts'][s]:.0f} deg")
        if a.get("power_offset_db", 0.0) != b.get("power_offset_db", 0.0):
            out.append(f"{lbl}: power offset {b.get('power_offset_db',0.0):+.0f} -> {a.get('power_offset_db',0.0):+.0f} dB")
    return out

def clean(x):
    return None if x is None or (isinstance(x, float) and not np.isfinite(x)) else round(float(x), 2)

In [ ]:
if "RESULT" not in globals():
    raise RuntimeError("This cell needs the setup cells above it. The kernel was restarted, or "
                       "cells 1-4 have not run yet: use Run > Run All Cells (or run cells 1-4 "
                       "first, then this one).")
# --- Capacity: the twin optimiser (Sionna RT + SYS link curve) -----------
if FUNCTION in ("capacity", "all"):
    policy = PKG["policy"]
    if MODE == "single":
        site = PKG["sites"][0]
        tx = [site["position_enu_m"][0], site["position_enu_m"][1],
              site["ground_z_m"] + site["height_m_agl"]]
        rx = twin_mod.street_receivers(PLY, tx[0], tx[1],
                                       half=site.get("eval_half_m", 150.0),
                                       step=site.get("eval_step_m", 20.0))
        scorer = twin_mod.TwinScorer(trace_for(site, rx, "site"), tx[:2])
        base = cfg_of(site)
        w_cur = twin_mod.traffic_weights(scorer, base, site["traffic_shares_current"])
        w_proj = twin_mod.traffic_weights(scorer, base, policy["traffic_shares_projected"])
        base_proj = scorer.score(base, w_proj)
        best, _, best_proj, best_cur, hist = optimise(scorer, base, w_proj, w_cur, policy)
        improved = best != base
        chosen = best if improved else base
        chosen_proj = best_proj if improved else base_proj
        print("AS-BUILT  (projected traffic):", fmt_k(base_proj))
        print("OPTIMISED (projected traffic):", fmt_k(chosen_proj))
        print("provenance:", "twin_guarded_accept" if improved else "no_improvement_keep_as_built",
              f"| candidates evaluated: {len(hist)}")
        Rb = scorer.sector_rsrp(base); Ra = scorer.sector_rsrp(chosen)
        mb = np.nanmax(np.where(np.isnan(Rb), -np.inf, Rb), axis=1)
        ma = np.nanmax(np.where(np.isnan(Ra), -np.inf, Ra), axis=1)
        RESULT.update({
            "configs_before": {site["site_id"]: base}, "configs_after": {site["site_id"]: chosen},
            "kpis_before": base_proj, "kpis_after": chosen_proj,
            "changes": describe_changes({site["site_id"]: base}, {site["site_id"]: chosen}),
            "provenance": "twin_guarded_accept" if improved else "no_improvement_keep_as_built",
            "maps": {"x": [clean(v) for v in rx[:, 0]], "y": [clean(v) for v in rx[:, 1]],
                     "before": [clean(v) for v in mb], "after": [clean(v) for v in ma]}})
    else:
        sites = PKG["sites"]
        rx = twin_mod.street_receivers(PLY, 0.0, 0.0, half=480.0, step=40.0)
        traces = {s["site_id"]: trace_for(s, rx, "cluster") for s in sites}
        scorer = ClusterScorer(traces, sites)
        base_configs = {s["site_id"]: cfg_of(s) for s in sites}
        totals = PKG["site_traffic_totals"]
        cur = {s["site_id"]: s["traffic_shares_current"] for s in sites}
        proj = policy["traffic_shares_projected"]
        w_cur = scorer.traffic_weights(base_configs, totals, cur)
        w_proj = scorer.traffic_weights(base_configs, totals, proj)
        base_proj = scorer.score(base_configs, w_proj)
        best, _, best_proj, best_cur, hist = optimise_cluster(scorer, base_configs, w_proj, w_cur, policy)
        improved = best != base_configs
        chosen = best if improved else base_configs
        chosen_proj = best_proj if improved else base_proj
        print("AS-BUILT  (projected traffic):", fmt_k(base_proj))
        print("OPTIMISED (projected traffic):", fmt_k(chosen_proj))
        print("provenance:", "twin_guarded_accept_cluster" if improved else "no_improvement_keep_as_built",
              f"| candidates evaluated: {len(hist)}")
        Rb = scorer.joint_rsrp(base_configs); Ra = scorer.joint_rsrp(chosen)
        mb = np.nanmax(np.where(np.isnan(Rb), -np.inf, Rb), axis=1)
        ma = np.nanmax(np.where(np.isnan(Ra), -np.inf, Ra), axis=1)
        RESULT.update({
            "configs_before": base_configs, "configs_after": chosen,
            "kpis_before": base_proj, "kpis_after": chosen_proj,
            "changes": describe_changes(base_configs, chosen),
            "provenance": "twin_guarded_accept_cluster" if improved else "no_improvement_keep_as_built",
            "per_site_served_coverage": {
                sid: {"before": base_proj["coverage_by_serving_site"].get(sid),
                      "after": chosen_proj["coverage_by_serving_site"].get(sid)}
                for sid in base_configs},
            "maps": {"x": [clean(v) for v in rx[:, 0]], "y": [clean(v) for v in rx[:, 1]],
                     "before": [clean(v) for v in mb], "after": [clean(v) for v in ma]}})
    for c in RESULT["changes"]:
        print(" change:", c)

In [ ]:
if "RESULT" not in globals():
    raise RuntimeError("This cell needs the setup cells above it. The kernel was restarted, or "
                       "cells 1-4 have not run yet: use Run > Run All Cells (or run cells 1-4 "
                       "first, then this one).")
# --- Slice / PRB: existing reward from the counters, SYS-scored choice ---
if FUNCTION in ("slice", "all"):
    import policy as pol
    from env_sys import SysCellEnv
    # The SYS environment scores every profile at each tower's own offered load
    # (its cell_load counter); towers at the same load share one scoring. Each tower
    # then judges the profiles against its own counters and current PRB shares.
    RESULT["changes"] = [x for x in RESULT["changes"] if "slice profile " not in x.lower()]
    RESULT["slices"] = []
    _scored = {}
    def profiles_at(load):
        key = round(float(load), 2)
        if key not in _scored:
            env = SysCellEnv(n_users=30, n_slots=100, seed=11)   # fresh, seeded: reproducible per load
            rows = []
            for i, (name, shares, sched) in enumerate(pol.PROFILES):
                k = env.evaluate(i, load=key)
                rows.append({"name": name, "shares": list(shares), "scheduler": sched,
                             "kpis": [round(float(v), 2) for v in k], "reward": round(pol.reward(k), 2)})
            print(f"SYS profiles at offered load {key:.2f}:")
            for r in rows:
                print(f"  {r['name']:16s} shares {r['shares']}  reward {r['reward']:.2f}  median DL "
                      f"{r['kpis'][0]:.1f}  edge DL {r['kpis'][2]:.1f}  fairness {r['kpis'][4]:.2f}")
            _scored[key] = rows
        return _scored[key]
    for site in PKG["sites"]:
        sid = site["site_id"]; lbl = site.get("label", sid)
        c = counters_for(sid)
        rows = profiles_at(min(1.0, max(0.05, float(c.get("load", 0.75)))))
        kpi_now = np.array([c["median_dl"], c["median_ul"], c["edge_dl"], c["edge_ul"],
                            c["fairness"], c["prb_util"], c["signalling"]], dtype=float)
        r_now = pol.reward(kpi_now)
        cur_idx = int(np.argmin([sum(abs(np.array(sh) - np.array(c["current_shares"])))
                                 for _, sh, _ in pol.PROFILES]))
        cur = rows[cur_idx]
        ok = [r for r in rows if r["reward"] > cur["reward"]
              and r["kpis"][2] >= 0.95 * cur["kpis"][2] and r["kpis"][4] >= cur["kpis"][4] - 0.02]
        chosen = max(ok, key=lambda r: r["reward"]) if ok else cur
        prov = "policy_guarded_accept" if chosen is not cur else "guarded_reject_keep_current_profile"
        print(f"{lbl}: EXISTING (from its counters) reward {r_now:.2f} | shares "
              f"{[round(v,2) for v in c['current_shares']]} | CHOSEN {chosen['name']} | provenance {prov}")
        RESULT["slices"].append({"site": sid, "load": round(float(c.get("load", 0.75)), 2),
                                 "counters_reward": round(r_now, 2), "profiles": rows,
                                 "current_profile": cur["name"], "chosen_profile": chosen["name"],
                                 "provenance": prov})
        if chosen is not cur:
            RESULT["changes"].append(
                f"{lbl}: slice profile {cur['name']} -> {chosen['name']}: PRB shares "
                f"{[round(v,2) for v in cur['shares']]} -> {[round(v,2) for v in chosen['shares']]}, "
                f"scheduler {chosen['scheduler']}")
    RESULT["slice"] = RESULT["slices"][0]   # the first tower, for readers of the one-tower format


In [ ]:
if "RESULT" not in globals():
    raise RuntimeError("This cell needs the setup cells above it. The kernel was restarted, or "
                       "cells 1-4 have not run yet: use Run > Run All Cells (or run cells 1-4 "
                       "first, then this one).")
# --- Audit: twin reference MCS vs the PM histogram (v4 section 10) -------
if FUNCTION in ("audit", "all"):
    import mcs_audit
    _run_ids = [x["site_id"] for x in PKG["sites"]]
    _pms = [p for p in (PKG.get("pm_sites") or [PKG["pm"]]) if p["site_id"] in _run_ids]
    if not _pms:
        raise ValueError(f"The audit has no MCS histograms for any site of this run ({_run_ids}).")
    RESULT["changes"] = [x for x in RESULT["changes"] if ": OLLA offset " not in x]
    RESULT["audits"] = []
    RESULT["cell_changes"] = []   # the same changes, structured (for the O1 check)
    for pm in _pms:
        site = next(s for s in PKG["sites"] if s["site_id"] == pm["site_id"])
        lbl = site.get("label", site["site_id"])
        print(f"--- MCS audit: {lbl}")
        tx = [site["position_enu_m"][0], site["position_enu_m"][1],
              site["ground_z_m"] + site["height_m_agl"]]
        rx = twin_mod.street_receivers(PLY, tx[0], tx[1],
                                       half=site.get("eval_half_m", 150.0),
                                       step=site.get("eval_step_m", 20.0))
        scorer = twin_mod.TwinScorer(trace_for(site, rx, "site"), tx[:2])  # same receivers as the single-site capacity run: reuse its trace
        sinr, serving, have = mcs_audit.receiver_sinr(scorer, cfg_of(site))
        cfgd = pm["gnb_configured"]
        sectors = []
        for s in range(3):
            m = have & (serving == s)
            hist = np.array(pm["mcs_histograms_by_sector"][str(s)], dtype=float)
            f = mcs_audit.audit_sector(scorer, sinr[m], hist,
                                       cfgd["olla_offset_db"], cfgd["mcs_cap"])
            sectors.append(f)
            rec = f["recommendation"]
            print(f"sector {s+1}: twin mean MCS {f['twin_mean_mcs']:.1f} | PM mean {f['pm_mean_mcs']:.1f} "
                  f"| fitted effective offset {f['estimated_effective_offset_db']:+.2f} dB (L1 {f['histogram_l1_distance']:.2f}) "
                  f"| correction {format(rec['olla_offset_adjustment_db'], '+.2f') + ' dB' if rec else 'none'} "
                  f"| twin-verified SE gain {100*f['se_gain_relative']:+.1f}%")
            if rec:
                RESULT["cell_changes"].append({"site": site["site_id"], "sector": s + 1, "param": "olla_offset_db",
                                               "from": cfgd["olla_offset_db"], "to": rec["new_configured_offset_db"],
                                               "source": "MCS audit"})
                if rec["mcs_cap"] != cfgd["mcs_cap"]:
                    RESULT["cell_changes"].append({"site": site["site_id"], "sector": s + 1, "param": "mcs_cap",
                                                   "from": cfgd["mcs_cap"], "to": rec["mcs_cap"], "source": "MCS audit"})
                RESULT["changes"].append(
                    f"{lbl} sector {s+1}: OLLA offset {cfgd['olla_offset_db']:+.2f} -> "
                    f"{rec['new_configured_offset_db']:+.2f} dB"
                    + (f", MCS cap {cfgd['mcs_cap']} -> {rec['mcs_cap']}" if rec["mcs_cap"] != cfgd["mcs_cap"] else ""))
        RESULT["audits"].append({"site": site["site_id"], "sectors": sectors})
    RESULT["audit"] = RESULT["audits"][0]  # kept for readers of the older one-site result
    RESULT["notes"].append(
        "Audit fit note: the histogram fit is sensitive to the trace realisation - on a "
        "sparse stand-in histogram like this one, fitted offsets can move by several dB "
        "between runs. Read the L1 distance as the fit's confidence (lower is better; a "
        "fit at the edge of the search grid is not trustworthy). The calibration of "
        "record is the controlled test, where the PM file is generated from the same "
        "twin state: a planted -3 dB offset was recovered exactly, and a healthy cell "
        "produced no action. The guardrails (1 dB action threshold, +/-4 dB bound, "
        ">=2% twin-verified SE gain) are what make acting on a fit safe.")
    print(RESULT["notes"][-1])

# --- Link audit: BLER / HARQ, MIMO rank, UL power control, load ----------
# Counter-based checks on the daily PM medians (not twin-verified). A BLER
# above target overrides an MCS-histogram fit that wants to raise OLLA on the
# same cell: the measured BLER is the stronger evidence.
if FUNCTION in ("audit", "all") and PKG.get("link_pm"):
    _run_ids = [x["site_id"] for x in PKG["sites"]]
    _labels = {x["site_id"]: x.get("label", x["site_id"]) for x in PKG["sites"]}
    _mcs_up = {}   # (label, sector) -> the OLLA change the MCS audit recommended there
    for _a in RESULT.get("audits", []):
        for _i, _sec in enumerate(_a["sectors"]):
            _rec = _sec.get("recommendation")
            if _rec:
                _mcs_up[(_labels.get(_a["site"], _a["site"]), _i + 1)] = _rec.get("olla_offset_adjustment_db", 0)
    RESULT["link_audits"] = []
    RESULT.setdefault("cell_changes", [])
    _olla_now = {p["site_id"]: p["gnb_configured"]["olla_offset_db"] for p in (PKG.get("pm_sites") or [PKG["pm"]])}
    for _site in PKG["link_pm"]:
        if _site["site_id"] not in _run_ids:
            continue
        _lbl = _labels[_site["site_id"]]
        _cells = []
        for _sk in sorted(_site["sectors"], key=int):
            _d = _site["sectors"][_sk]; _m = _d["median"]; _c = _d["config"]; _s = int(_sk)
            _f = []
            # 1. BLER / HARQ against the BLER target
            _ex = _m["dl_bler_initial_pct"] - _c["dl_bler_target_pct"]
            if _ex > 3 or _m["dl_bler_residual_pct"] > 1.0:
                _step = min(3.0, max(0.5, round(_ex / 4 * 2) / 2)) if _ex > 3 else 0.0
                _rec = []
                if _step:
                    _rec.append(f"lower the OLLA offset by {_step:.1f} dB (more conservative MCS)")
                if _m["dl_bler_residual_pct"] > 1.0:
                    _rec.append("residual BLER above 1 %: check HARQ max transmissions and RLC retransmissions")
                _conf = _mcs_up.get((_lbl, _s))
                if _conf is not None and _step:
                    # one OLLA change per cell: the measured BLER replaces the histogram fit
                    RESULT["changes"] = [x for x in RESULT["changes"] if not x.startswith(f"{_lbl} sector {_s}: OLLA offset")]
                    RESULT["cell_changes"] = [x for x in RESULT["cell_changes"]
                                              if not (x["site"] == _site["site_id"] and x["sector"] == _s and x["source"] == "MCS audit")]
                    _rec.append(f"replaces the MCS-histogram OLLA change ({_conf:+.2f} dB): measured BLER is the stronger evidence"
                                if _conf <= 0 else
                                f"MCS-histogram change to raise OLLA ({_conf:+.2f} dB) withdrawn: measured BLER is above target")
                _f.append({"area": "BLER / HARQ", "severity": "high" if _ex > 7 or _m["dl_bler_residual_pct"] > 2 else "medium",
                           "evidence": f"DL initial BLER {_m['dl_bler_initial_pct']:.1f} % (target {_c['dl_bler_target_pct']:.0f} %), "
                                       f"residual {_m['dl_bler_residual_pct']:.2f} %, HARQ avg tx {_m['harq_dl_avg_tx']:.2f}",
                           "recommendation": "; ".join(_rec)})
                if _step:
                    _o = float(_olla_now.get(_site["site_id"], 0.0))
                    RESULT["cell_changes"].append({"site": _site["site_id"], "sector": _s, "param": "olla_offset_db",
                                                   "from": _o, "to": round(_o - _step, 2), "source": "link audit (BLER)"})
                    RESULT["changes"].append(f"{_lbl} sector {_s}: OLLA offset -{_step:.1f} dB (DL BLER {_m['dl_bler_initial_pct']:.1f} % vs target {_c['dl_bler_target_pct']:.0f} %)")
            # 2. MIMO rank on a multi-layer cell
            if _c["mimo_layers_max"] >= 2 and _m["dl_rank1_pct"] > 50:
                _f.append({"area": "MIMO", "severity": "high" if _m["dl_rank1_pct"] > 60 else "medium",
                           "evidence": f"rank 1 on {_m['dl_rank1_pct']:.0f} % of DL transmissions, average rank {_m['dl_avg_rank']:.2f} "
                                       f"on a {_c['mimo_layers_max']}-layer cell",
                           "recommendation": "field check: antenna port / cross-pol jumper swap, CSI-RS port configuration, rank restriction"})
            # 3. Uplink power control
            if _m["ul_phr_negative_pct"] > 25:
                _dp = min(6, max(2, round((_m["ul_phr_negative_pct"] - 15) / 5)))
                _newp0 = _c["p0_nominal_pusch_dbm"] - _dp
                _f.append({"area": "UL power control", "severity": "high" if _m["ul_phr_negative_pct"] > 35 else "medium",
                           "evidence": f"{_m['ul_phr_negative_pct']:.0f} % of UEs power-limited (negative PHR), UL BLER {_m['ul_bler_initial_pct']:.1f} %, "
                                       f"UL SINR {_m['ul_sinr_avg_db']:.1f} dB, P0 {_c['p0_nominal_pusch_dbm']:.0f} dBm, alpha {_c['pusch_alpha']}",
                           "recommendation": f"lower P0 nominal PUSCH by {_dp} dB to {_newp0:.0f} dBm"})
                RESULT["cell_changes"].append({"site": _site["site_id"], "sector": _s, "param": "p0_nominal_pusch_dbm",
                                               "from": int(_c["p0_nominal_pusch_dbm"]), "to": int(_newp0),
                                               "source": "link audit (UL power control)"})
                RESULT["changes"].append(f"{_lbl} sector {_s}: P0 nominal PUSCH {_c['p0_nominal_pusch_dbm']:.0f} -> {_newp0:.0f} dBm")
            # 4. Load / congestion
            if _d["congested_days_pct"] > 30 or _m["prb_util_dl_pct"] > 85:
                _f.append({"area": "Load", "severity": "high" if _d["congested_days_pct"] > 50 else "medium",
                           "evidence": f"congested on {_d['congested_days_pct']:.0f} % of {_d['days']} days, PRB median {_m['prb_util_dl_pct']:.0f} %, "
                                       f"user DL {_m['user_dl_tput_mbps']:.1f} Mbps",
                           "recommendation": "capacity, not link adaptation: see the Capacity and Slice / PRB results; consider an extra carrier"})
            _cells.append({"sector": _s, "days": _d["days"], "findings": _f})
            for _x in _f:
                print(f"{_lbl} sector {_s} [{_x['area']}, {_x['severity']}]: {_x['evidence']} -> {_x['recommendation']}")
            if not _f:
                print(f"{_lbl} sector {_s}: link counters within limits")
        RESULT["link_audits"].append({"site": _site["site_id"], "cells": _cells})


In [ ]:
if "RESULT" not in globals():
    raise RuntimeError("This cell needs the setup cells above it. The kernel was restarted, or "
                       "cells 1-4 have not run yet: use Run > Run All Cells (or run cells 1-4 "
                       "first, then this one).")
# --- Energy: sleep-window plan from the counters (anchored, v4 sec. 11) --
if FUNCTION in ("energy", "all"):
    # Each tower is planned from its own counters. A congested tower (PRB use >= 85 %)
    # keeps every carrier on: sleeping one would cut capacity where it is short.
    # A tower whose daily link counters show it congested on >= 30 % of days is
    # also kept fully on, whatever its busy-hour PRB counter says.
    CONGESTED_PRB, CONGESTED_DAYS = 85.0, 30.0
    _cong = {}
    for _lp in (PKG.get("link_pm") or []):
        _d = [v.get("congested_days_pct", 0.0) for v in _lp["sectors"].values()]
        if _d:
            _cong[_lp["site_id"]] = sum(_d) / len(_d)
    plan, per_site = [], []
    for s in PKG["sites"]:
        sid = s["site_id"]; lbl = s.get("label", sid)
        c = counters_for(sid)
        util = float(c.get("prb_util", 81.0)); low = float(c.get("low_hours", 6))
        carriers = int(c.get("carriers_per_site", 2))
        cdays = _cong.get(sid)
        if util >= CONGESTED_PRB or carriers < 2 or (cdays is not None and cdays >= CONGESTED_DAYS):
            saving, sleep = 0.0, False
            why = (f"PRB use {util:.0f} % (congested)" if util >= CONGESTED_PRB else
                   "a single carrier" if carriers < 2 else
                   f"its link counters show it congested on {cdays:.0f} % of days")
            plan.append(f"{lbl}: no carrier sleep - {why}; all {carriers} carriers stay on")
        else:
            saving = min(12.0, max(2.0, 7.21 * (low / 6.0) * (1.0 if util <= 88 else 0.85)))
            sleep = True
            plan.append(f"{lbl}: sleep {carriers - 1} of {carriers} carriers for {low:.0f} "
                        f"low-traffic hours/night, 1 carrier kept on (PRB use {util:.0f} %); slice "
                        f"floors and the twin coverage guardrail apply to any sleep decision")
        per_site.append({"site": sid, "prb_util": util, "congested_days_pct": None if cdays is None else round(cdays, 1), "low_hours": low, "carriers": carriers,
                         "sleep": sleep, "saving_pct": round(saving, 1),
                         "index_after": round(100.0 - saving, 1)})
        print(f"{lbl}: energy index 100 -> {100.0 - saving:.1f} (saving {saving:.1f}%"
              + (f": RANOpt-validated +7.21% scaled by {low:.0f} low-traffic hours and utilisation {util:.0f}%)"
                 if sleep else ": not slept)"))
    saving = sum(p["saving_pct"] for p in per_site) / len(per_site)   # every site counted alike
    # re-running this cell replaces the previous energy plan in the change list
    RESULT["changes"] = [x for x in RESULT["changes"]
                         if x not in RESULT.get("energy", {}).get("plan", [])] + plan
    RESULT["energy"] = {"saving_pct": round(saving, 1), "index_before": 100.0,
                        "index_after": round(100.0 - saving, 1), "plan": plan, "per_site": per_site}
    if len(per_site) > 1:
        print(f"cluster energy index 100 -> {100.0 - saving:.1f} (mean saving over the towers)")


In [ ]:
if "RESULT" not in globals():
    raise RuntimeError("This cell needs the setup cells above it. The kernel was restarted, or "
                       "cells 1-4 have not run yet: use Run > Run All Cells (or run cells 1-4 "
                       "first, then this one).")
# --- A1 check: the slice and energy decisions as A1 policies -------------
# Each tower's slice profile and carrier-sleep plan becomes an A1 policy (rApp-defined
# policy types). They are checked against their JSON schemas here, then sent to the
# O-RAN SC A1 simulator (A1-P v2, standing in for a Near-RT RIC): create, read back,
# status, delete, plus one invalid policy that must be rejected. The simulator only
# stores and acknowledges policies - it shows the policies are well-formed and
# accepted over A1, not their effect on the network. Capacity, OLLA and P0 changes
# are O1 configuration changes, so they are not part of this check.
RESULT.pop("a1", None)
if globals().get("A1_CHECK", True) and (RESULT.get("slices") or RESULT.get("energy")):
    import a1_check
    _pols = a1_check.build_policies(RESULT)
    _local = a1_check.validate_locally(_pols)
    A1 = {"policy_types": a1_check.POLICY_TYPES, "local_schema_check": _local,
          "local_ok": all(v is None for v in _local.values())}
    print(f"A1 policies: {len(_pols)} ({', '.join(p['policy_id'] for p in _pols)})")
    print("schema check on this machine:", "all valid" if A1["local_ok"] else _local)
    try:
        A1["sim"] = a1_check.run_check(_pols)
        for r in A1["sim"]["policies"]:
            print(f"  {r['policy_id']:22s} create {r['create']}  read back {r['read_back']}"
                  f"{' (same)' if r['read_back_matches'] else ' (DIFFERENT)'}  status {r['status_code']} "
                  f"{r['status']}  delete {r['delete']}  -> {'OK' if r['ok'] else 'FAILED'}")
        n = A1["sim"]["negative"]
        print(f"  invalid policy ({n['what']}): HTTP {n['http']} -> {'rejected, as it should be' if n['rejected'] else 'NOT rejected'}")
        print("A1 check:", "passed" if A1["sim"]["all_ok"] else "FAILED", "-", A1["sim"]["endpoint"])
    except Exception as _e:
        A1["sim_error"] = str(_e)[:400]
        print("A1 simulator not run:", A1["sim_error"])
        print("(the schema check above still holds; the A1 exchange needs internet on first use)")
    RESULT["a1"] = A1
elif not globals().get("A1_CHECK", True):
    print("A1 check off (A1_CHECK = False in the settings cell)")
else:
    print("A1 check: nothing to send - run slice, energy or all to get A1 policies")


In [ ]:
if "RESULT" not in globals():
    raise RuntimeError("This cell needs the setup cells above it. The kernel was restarted, or "
                       "cells 1-4 have not run yet: use Run > Run All Cells (or run cells 1-4 "
                       "first, then this one).")
# --- O1 check: the configuration changes as O1 edits, against YANG ---------
# Each recommended configuration change becomes an O1 edit: antenna tilt, azimuth and
# transmit power on 3GPP NR NRM attributes (TS 28.541 / 28.662, Rel-18 YANG), and OLLA
# offset, MCS cap and P0 on a vendor-style module (3GPP has no attribute for them).
# Every edited leaf must exist in the models, be writable, and its value must fit the
# leaf's type and range; two out-of-range edits must be rejected. This shows the edits
# are well-formed - not that a real gNB accepts them (no NETCONF exchange here).
RESULT.pop("o1", None)
if globals().get("O1_CHECK", True) and (RESULT.get("configs_after") or RESULT.get("cell_changes")):
    import o1_check
    _edits = o1_check.build_edits(RESULT)
    if not _edits:
        print("O1 check: no configuration change to send (the twin gate kept the as-built configuration)")
        RESULT["o1"] = {"edits": [], "all_ok": True, "note": "no configuration change"}
    else:
        try:
            O1 = o1_check.run_check(_edits)
            print("O1 models:", O1["models"])
            for e in O1["edits"]:
                print(f"  {e['instance']:78s} = {e['value']!s:>7s} {e['units']:10s} "
                      f"{'OK' if e['ok'] else 'FAILED: ' + e['check']}  [{'3GPP' if e['standard'] else 'vendor'}]")
            for n in O1["negative"]:
                print(f"  invalid edit ({n['what']}): {'rejected - ' + n['reason'] if n['rejected'] else 'NOT rejected'}")
            print("O1 check:", "passed" if O1["all_ok"] else "FAILED",
                  f"- {len(O1['edits'])} edits, {sum(e['standard'] for e in O1['edits'])} on 3GPP attributes, "
                  f"{sum(not e['standard'] for e in O1['edits'])} vendor-specific")
            RESULT["o1"] = O1
        except Exception as _e:
            RESULT["o1"] = {"edits": _edits, "error": str(_e)[:400]}
            print("O1 check not run:", RESULT["o1"]["error"])
elif not globals().get("O1_CHECK", True):
    print("O1 check off (O1_CHECK = False in the settings cell)")
else:
    print("O1 check: nothing to check - run capacity, audit or all to get configuration changes")


In [ ]:
if "RESULT" not in globals():
    raise RuntimeError("This cell needs the setup cells above it. The kernel was restarted, or "
                       "cells 1-4 have not run yet: use Run > Run All Cells (or run cells 1-4 "
                       "first, then this one).")
# --- The result: read it here -------------------------------------------
# Everything the run concluded is printed in this cell's output: the
# recommended changes, the before -> after numbers and the RSRP maps.
# Nothing is written to a file and nothing is downloaded.
R = RESULT
print("=" * 66)
print("RESULT - mode:", R.get("mode"), "| function:", R.get("function"))
# one provenance per function that ran ("all" runs several)
parts = [R["provenance"]] if R.get("provenance") else []
for _s in (R.get("slices") or ([R["slice"]] if R.get("slice") else [])):
    parts.append(site_label(_s.get("site", "")) + " " + _s.get("provenance", "policy_guarded_accept"))
if R.get("audit"):
    parts.append("mcs_audit_guarded (per-sector detail below)")
if R.get("energy"):
    parts.append("energy_anchored_estimate")
prov = ", ".join(parts) if parts else "-"
print("Provenance:", prov)
if R.get("changes"):
    print("\nRecommended changes:")
    for c in R["changes"]:
        print("  -", c)
kb, ka = R.get("kpis_before"), R.get("kpis_after")
if kb and ka:
    print("\nBefore -> after (twin-verified):")
    for lbl, k, d in [("Coverage (%)", "coverage_pct", 1),
                      ("Median RSRP (dBm)", "median_rsrp", 1),
                      ("Edge RSRP p5 (dBm)", "edge_rsrp_p5", 1),
                      ("Median SINR (dB)", "median_sinr", 1),
                      ("Path availability (%)", "path_pct", 1),
                      ("Weighted SE (bit/s/Hz)", "capacity_weighted_se", 2)]:
        if k in kb and k in ka:
            print(f"  {lbl:26s} {kb[k]:>8.{d}f} -> {ka[k]:>8.{d}f}")
    if R.get("per_site_served_coverage"):
        print("\nPer-site served coverage (%):")
        for s, v in R["per_site_served_coverage"].items():
            f1 = lambda x: "-" if x is None else f"{x:.1f}"
            print(f"  {s}: {f1(v['before'])} -> {f1(v['after'])}")
for s in (R.get("slices") or ([R["slice"]] if R.get("slice") else [])):
    profs = {q["name"]: q for q in s["profiles"]}
    chosen = profs[s["chosen_profile"]]
    print("\nSlice / PRB -", site_label(s.get("site", "")) + ":", s["current_profile"], "->",
          s["chosen_profile"], "| scheduler:", chosen["scheduler"])
    print("  Reward - this tower's counters, existing shares:", round(s["counters_reward"], 2),
          "| SYS scored at offered load", s.get("load", 0.75))
    print(f"  {'profile':16s} {'shares eMBB/URLLC/mMTC':24s} {'twin reward':>11s}")
    for q in s["profiles"]:
        mark = "  <- chosen" if q["name"] == s["chosen_profile"] else ("  (existing)" if q["name"] == s["current_profile"] else "")
        print(f"  {q['name']:16s} {str(q['shares']):24s} {q['reward']:>11.2f}{mark}")
for a in (R.get("audits") or ([R["audit"]] if R.get("audit") else [])):
    print("\nMCS audit - site:", a.get("site"))
    print(f"  {'sector':7s} {'offset':>7s} {'correct':>8s} {'SE gain':>8s} {'fit L1':>7s}")
    for i2, sec in enumerate(a["sectors"]):
        rec = sec.get("recommendation") or {}  # None when the sector needs no action
        print(f"  {i2 + 1:<7d} {sec['estimated_effective_offset_db']:>7.2f} "
              f"{rec.get('olla_offset_adjustment_db', 0):>+8.2f} "
              f"{sec['se_gain_relative'] * 100:>7.1f}% {sec['histogram_l1_distance']:>7.2f}")
        if sec.get("cap_binding") and rec:
            print("    MCS cap is binding in the twin - the recommendation raises the cap to",
                  rec.get("mcs_cap"))
for la in R.get("link_audits", []):
    print("\nLink audit (counter-based) - site:", la["site"])
    for c in la["cells"]:
        if not c["findings"]:
            print(f"  sector {c['sector']}: within limits")
        for x in c["findings"]:
            print(f"  sector {c['sector']} [{x['area']}, {x['severity']}] {x['evidence']}")
            print(f"      -> {x['recommendation']}")
if R.get("energy"):
    e = R["energy"]
    print(f"\nEnergy index: {e['index_before']:.0f} -> {e['index_after']:.1f} "
          f"(estimated saving {e['saving_pct']:.1f}% - anchored on the RANOpt energy "
          f"validation over each site's low-traffic hours; an estimate, not a twin trace)")
    for ps in e.get("per_site", []):
        print(f"  {site_label(ps['site'])}: PRB {ps['prb_util']:.0f} %, {ps['low_hours']:.0f} low hours, "
              f"{ps['carriers']} carriers -> index {ps['index_after']:.1f}" + ("" if ps["sleep"] else " (no sleep)"))
    for pl in e.get("plan", []):
        print("  -", pl)
if R.get("a1"):
    a = R["a1"]
    print("\nA1 check (slice and energy decisions as A1 policies):")
    print("  schema check on this machine:", "all valid" if a.get("local_ok") else a.get("local_schema_check"))
    if a.get("sim"):
        s = a["sim"]
        print("  endpoint:", s["endpoint"])
        for r in s["policies"]:
            print(f"  {r['policy_id']:22s} {r['type']:30s} create {r['create']} read {r['read_back']} "
                  f"status {r['status']} delete {r['delete']} -> {'OK' if r['ok'] else 'FAILED'}")
        print(f"  invalid policy rejected: {s['negative']['rejected']} (HTTP {s['negative']['http']})")
        print("  A1 check:", "passed" if s["all_ok"] else "FAILED")
    elif a.get("sim_error"):
        print("  simulator not run:", a["sim_error"])
if R.get("o1"):
    o = R["o1"]
    print("\nO1 check (configuration changes as O1 edits, against YANG):")
    if o.get("error"):
        print("  not run:", o["error"])
    elif not o.get("edits"):
        print("  no configuration change to check")
    else:
        print("  models:", o["models"])
        for e in o["edits"]:
            print(f"  {e['instance']:78s} = {e['value']!s:>7s} {e['units']:10s} {'OK' if e['ok'] else 'FAILED'}"
                  f"  [{'3GPP' if e['standard'] else 'vendor'}]")
        for n in o["negative"]:
            print(f"  invalid edit rejected: {n['rejected']} ({n['what']})")
        print("  O1 check:", "passed" if o["all_ok"] else "FAILED")
for n in R.get("notes", []):
    print("Note:", n)
print("=" * 66)

maps = R.get("maps")
if maps:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for ax, key, ttl in [(axes[0], "before", "Before - best-server RSRP (dBm per RE)"),
                         (axes[1], "after", "After - best-server RSRP (dBm per RE)")]:
        pts = [(x, y, v) for x, y, v in zip(maps["x"], maps["y"], maps[key]) if v is not None]
        sc = ax.scatter([q[0] for q in pts], [q[1] for q in pts],
                        c=[q[2] for q in pts], cmap="turbo", vmin=-140, vmax=-60, s=16)
        ax.set_title(ttl, fontsize=9)
        ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
    fig.colorbar(sc, ax=axes, shrink=0.85, label="dBm per RE")
    plt.show()
